# 01 · Load and clean

**Goal:** turn the seven yearly CSV files from Rijden de Treinen (2019–2025) into one clean, typed table,
and record every data problem found along the way.

- **Input:** `data/raw/disruptions-<year>.csv` (created by `src/download.py`)
- **Output:** `data/processed/disruptions_clean.parquet`
- **Unit of analysis:** one row = one disruption message published by NS.
- **Target (built in notebook 02):** derived from `duration_minutes`.

Rule for this notebook: only remove rows that are clearly unusable. Judgement calls (threshold,
merging related records, which causes to model) are left for notebook 02.

## 1. Load the raw files

In [1]:
from pathlib import Path

import pandas as pd

repo_root = Path.cwd().parent
raw_dir = repo_root / "data" / "raw"
processed_dir = repo_root / "data" / "processed"

print("Raw data folder:", raw_dir)
print("Folder exists:  ", raw_dir.exists())

Raw data folder: d:\CV and applications\improving\rail-disruptions-nl\data\raw
Folder exists:   True


In [2]:
filepaths = sorted(raw_dir.glob("disruptions-*.csv"))
dfs = [pd.read_csv(path) for path in filepaths]
df = pd.concat(dfs, ignore_index=True)

print([path.name for path in filepaths])
df.shape

['disruptions-2019.csv', 'disruptions-2020.csv', 'disruptions-2021.csv', 'disruptions-2022.csv', 'disruptions-2023.csv', 'disruptions-2024.csv', 'disruptions-2025.csv']


(38571, 14)

In [3]:
df.head()

,rdt_id,ns_lines,rdt_lines,rdt_lines_id,rdt_station_names,rdt_station_codes,cause_nl,cause_en,statistical_cause_nl,statistical_cause_en,cause_group,start_time,end_time,duration_minutes
0,25236,Utrecht-'s-Hertogenbosch,'s-Hertogenbosch - Utrecht Centraal,151,"Geldermalsen,'s-Hertogenbosch,Zaltbommel","GDM, HT, ZBM",defecte bovenleiding,damaged overhead wires,defecte bovenleiding,damaged overhead wires,infrastructure,2019-01-01 06:32:16,2019-01-01 06:33:31,1.0
1,25237,Almere-Lelystad; Almere-Lelystad; Hoofddorp-Sc...,"Almere Oostvaarders - Utrecht Centraal, Amster...","144,145,149","Almere Oostvaarders,Lelystad Centrum","ALMO, LLS",vandalisme,vandalism,vandalisme,vandalism,external,2019-01-01 07:12:00,2019-01-01 09:53:48,162.0
2,25238,Amersfoort-Zwolle,Amersfoort - Zwolle,48,"Ermelo,Harderwijk,Putten","EML, HD, PT",seinstoring,signal failure,seinstoring,signal failure,infrastructure,2019-01-01 08:26:58,2019-01-01 08:44:01,17.0
3,25239,Utrecht-'s-Hertogenbosch; Utrecht-Tiel,"'s-Hertogenbosch - Utrecht Centraal, Tiel - Ut...","150,151","Culemborg,Geldermalsen,Houten Castellum","CL, GDM, HTNC",inzet van hulpdiensten,an emergency call,inzet van hulpdiensten,an emergency call,external,2019-01-01 08:36:00,2019-01-01 09:09:43,34.0
4,25240,Amersfoort-Zwolle,Amersfoort - Zwolle,48,"Amersfoort Centraal,Amersfoort Schothorst,Amer...","AMF, AMFS, AVAT, EML, HD, HDE, NKK, NS, PT, WZ...",herstelwerkzaamheden,repair works,herstelwerkzaamheden,repair works,engineering work,2019-01-01 09:45:48,2019-01-01 12:47:54,182.0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 38571 entries, 0 to 38570
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   rdt_id                38571 non-null  int64  
 1   ns_lines              38571 non-null  str    
 2   rdt_lines             38559 non-null  str    
 3   rdt_lines_id          38559 non-null  str    
 4   rdt_station_names     38563 non-null  str    
 5   rdt_station_codes     38563 non-null  str    
 6   cause_nl              38571 non-null  str    
 7   cause_en              38571 non-null  str    
 8   statistical_cause_nl  38571 non-null  str    
 9   statistical_cause_en  38571 non-null  str    
 10  cause_group           38569 non-null  str    
 11  start_time            38571 non-null  str    
 12  end_time              38461 non-null  str    
 13  duration_minutes      38460 non-null  float64
dtypes: float64(1), int64(1), str(12)
memory usage: 14.1 MB


## 2. First look: missing values, unique values, IDs

In [5]:
df.isna().sum()

rdt_id                    0
ns_lines                  0
rdt_lines                12
rdt_lines_id             12
rdt_station_names         8
rdt_station_codes         8
cause_nl                  0
cause_en                  0
statistical_cause_nl      0
statistical_cause_en      0
cause_group               2
start_time                0
end_time                110
duration_minutes        111
dtype: int64

In [6]:
df.nunique()

rdt_id                  38571
ns_lines                 1517
rdt_lines                 518
rdt_lines_id              518
rdt_station_names        1951
rdt_station_codes        1951
cause_nl                   95
cause_en                   91
statistical_cause_nl       95
statistical_cause_en       91
cause_group                 9
start_time              38523
end_time                38410
duration_minutes         1537
dtype: int64

In [7]:
df["rdt_id"].is_unique

True

**Observations**

- 38,571 rows over 7 years, 14 columns. `rdt_id` is unique, so no disruption appears twice across files.
- `end_time` is missing in 110 rows and `duration_minutes` in 111: one row has an end time but no duration (section 3).
- `rdt_lines` / `rdt_lines_id` are missing in 12 rows and station names in 8. These are features, not the target,
  so the rows are kept and the gaps are filled in notebook 03.
- `cause_group` is missing in 2 rows (section 7).
- `start_time` has 38,523 unique values for 38,571 rows, so some messages start in the same second (section 9).

## 3. Timestamps and duration

In [8]:
df["start_time"] = pd.to_datetime(df["start_time"])
df["end_time"] = pd.to_datetime(df["end_time"])

df[["start_time", "end_time"]].dtypes

start_time    datetime64[us]
end_time      datetime64[us]
dtype: object

Recompute the duration from the timestamps and compare it with RDT's `duration_minutes`,
to check what the column really contains.

In [9]:
df["calculated_duration"] = (df["end_time"] - df["start_time"]).dt.total_seconds() / 60

diff = df["calculated_duration"] - df["duration_minutes"]
diff.describe()

count    38460.000000
mean        -0.000494
std          0.860853
min        -59.933333
25%         -0.250000
50%          0.000000
75%          0.250000
max         60.466667
dtype: float64

Does RDT round the duration to the nearest minute, or cut off the seconds?
The comparison is done only on rows where `duration_minutes` exists, because `NaN == NaN` is always `False`.

In [10]:
has_duration = df["duration_minutes"].notna()

share_round = (df.loc[has_duration, "calculated_duration"].round() == df.loc[has_duration, "duration_minutes"]).mean()
share_floor = ((df.loc[has_duration, "calculated_duration"] // 1) == df.loc[has_duration, "duration_minutes"]).mean()

print(f"Matches when rounding:       {share_round:.1%}")
print(f"Matches when cutting seconds: {share_floor:.1%}")

Matches when rounding:       99.2%
Matches when cutting seconds: 51.2%


In [11]:
# The remaining mismatches after rounding: are they exactly x.5 minutes?
not_matching = has_duration & (df["calculated_duration"].round() != df["duration_minutes"]) & (diff.abs() <= 1)
print("Mismatches of at most 1 minute:", not_matching.sum())
(df.loc[not_matching, "calculated_duration"] % 1).round(3).value_counts()

Mismatches of at most 1 minute: 289


calculated_duration
0.5    289
Name: count, dtype: int64

`duration_minutes` is rounded to the nearest minute. The few hundred remaining mismatches are all exactly
x.5 minutes: pandas rounds 0.5 to the nearest even number (2.5 → 2), RDT rounds it up (2.5 → 3).

Now the rows where the difference is larger than one minute:

In [12]:
(diff.abs() > 1).sum()

np.int64(7)

In [13]:
df.loc[diff.abs() > 1, ["rdt_id", "start_time", "end_time", "calculated_duration", "duration_minutes"]]

,rdt_id,start_time,end_time,calculated_duration,duration_minutes
21762,46998,2023-03-13 16:04:47,2023-04-11 10:04:04,41399.283333,41339.0
21886,47122,2023-03-21 13:03:35,2023-03-29 00:56:21,10792.766667,10733.0
21905,47141,2023-03-23 06:32:19,2023-03-28 09:26:47,7374.466667,7314.0
27379,52768,2024-03-30 05:05:05,2024-03-31 17:44:25,2199.333333,2139.0
30763,56152,2024-10-26 05:54:45,2024-10-27 02:02:51,1208.100000,1268.0
33573,58962,2025-03-24 07:26:11,2025-04-01 00:37:32,11111.350000,11051.0
37378,62767,2025-10-25 00:47:01,2025-10-27 23:52:05,4265.066667,4325.0


**Daylight saving time.** All 7 disruptions were still running during a clock change:

- last Sunday of **March**: clocks jump 02:00 → 03:00, so the day has 23 hours → my calculation is **60 min too long**;
- last Sunday of **October**: clocks go back 03:00 → 02:00, so the day has 25 hours → my calculation is **60 min too short**.

The timestamps are local clock times without a timezone, so subtracting them ignores the clock change.
RDT's `duration_minutes` already handles it, so **`duration_minutes` is the correct column** and is used as the target.
`calculated_duration` was only a check.

The one row that has an `end_time` but no `duration_minutes`:

In [14]:
df.loc[df["end_time"].notna() & df["duration_minutes"].isna(),
       ["rdt_id", "rdt_lines", "cause_en", "start_time", "end_time", "calculated_duration"]]

,rdt_id,rdt_lines,cause_en,start_time,end_time,calculated_duration
8223,33459,"Amersfoort - Utrecht Centraal, Baarn - Utrecht...",broken down train,2020-07-23 20:29:13,2020-09-21 08:24:24,85675.183333


A "broken down train" lasting about 59 days is not believable. This looks like a message that was never closed
properly, and RDT left the duration empty on purpose. It is **not filled in**: it is dropped together with the rows
that have no end time.

## 4. Missing end times

In [15]:
no_end = df["end_time"].isna()
print("Rows without end_time:", no_end.sum())
df.loc[no_end, "start_time"].dt.year.value_counts().sort_index()

Rows without end_time: 110


start_time
2019     3
2020    58
2021    13
2022    15
2023     8
2024    11
2025     2
Name: count, dtype: int64

More than half of the missing end times (58 of 110) are in 2020, which points to a logging problem that year.

**Decision: drop.** Without an end time there is no duration, so there is no target. A row without a target
cannot be used for training or testing, and the target must never be imputed.

## 5. Zero and negative durations

In [16]:
zero_duration = df["duration_minutes"] == 0
exact_zero = df["calculated_duration"] == 0

print("duration_minutes == 0:          ", zero_duration.sum())
print("  of which start == end exactly:", exact_zero.sum())
print("negative durations:             ", (df["calculated_duration"] < 0).sum())

duration_minutes == 0:           683
  of which start == end exactly: 455
negative durations:              0


In [17]:
df.loc[zero_duration, ["rdt_id", "rdt_lines", "cause_en", "start_time", "end_time"]].head()

,rdt_id,rdt_lines,cause_en,start_time,end_time
6,25242,Amsterdam Centraal - Woerden,broken down train,2019-01-01 11:28:46,2019-01-01 11:28:46
59,25295,"Brussel-Zuid/Midi - Rotterdam Centraal (HSL), ...",broken down train,2019-01-05 15:51:50,2019-01-05 15:51:50
101,25337,'s-Hertogenbosch - Eindhoven,points failure,2019-01-08 08:19:30,2019-01-08 08:19:30
107,25343,Amersfoort - Zwolle,broken down train,2019-01-08 12:51:40,2019-01-08 12:51:40
158,25394,'s-Hertogenbosch - Eindhoven,defective railway track,2019-01-10 17:39:09,2019-01-10 17:39:09


In [18]:
df.loc[zero_duration, "start_time"].dt.year.value_counts().sort_index()

start_time
2019    373
2020    120
2021      2
2022     46
2023     30
2024     44
2025     68
Name: count, dtype: int64

- 683 rows round to 0 minutes. In 455 of them the start and end time are the **same second**: the message was
  opened and closed at once (sent by mistake, withdrawn, or replaced). The other 228 lasted under 30 seconds.
- They are concentrated in 2019–2020 (493 of 683) and almost disappear in 2021 (2 rows). Real breakdowns do not
  vanish for a year, so this reflects **how messages were recorded**, not how trains behaved.
- Keeping them would label them "short disruption", which is a wrong label: they are not disruptions at all.
  Because they sit mostly in training years, they would also make the training years look different from
  2024–2025.

**Decision: drop all 683 rows with `duration_minutes == 0`.** Rows of 1–2 minutes are kept; the cut-off at 0 is a
judgement call and is documented here. No negative durations exist.

## 6. Shape of the duration distribution

In [19]:
df["duration_minutes"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])

count    38460.000000
mean       162.620281
std        859.044243
min          0.000000
50%         47.000000
75%        131.000000
90%        272.000000
95%        516.000000
99%       1438.410000
max      45302.000000
Name: duration_minutes, dtype: float64

In [20]:
(
    df.assign(duration_days=df["duration_minutes"] / (24 * 60))
    .sort_values("duration_days", ascending=False)
    [["rdt_id", "rdt_lines", "cause_en", "start_time", "duration_days"]]
    .head(10)
)

,rdt_id,rdt_lines,cause_en,start_time,duration_days
18389,43625,Roermond - Sittard,damaged railway bridge,2022-07-16 05:29:58,31.459722
15286,40522,"Berlin Ostbahnhof - Hengelo, Bielefeld Hbf - H...",problems with the rolling stock,2022-01-03 09:07:30,30.566667
21762,46998,Leeuwarden - Stavoren,repair works,2023-03-13 16:04:47,28.707639
21272,46508,"Groningen - Leeuwarden, Harlingen Haven - Leeu...",strike of Arriva staff,2023-02-08 04:48:23,27.846528
21273,46509,"Delfzijl - Groningen, Groningen - Bad Nieuwesc...",strike of Arriva staff,2023-02-08 04:48:27,27.846528
21276,46512,Groningen - Leeuwarden,strike of Arriva staff,2023-02-08 07:29:20,26.995833
33843,59232,"Liège-Guillemins - Maastricht, Maastricht - Ma...",staff strikes abroad,2025-04-08 05:18:14,21.771528
31471,56860,Heerlen - Maastricht,problems with the rolling stock,2024-12-03 10:15:01,21.583333
31832,57221,Heerlen - Maastricht,problems with the rolling stock,2024-12-27 07:45:33,19.678472
15199,40435,"Amsterdam Centraal - Schiphol Airport, Brussel...",large crowds,2021-12-28 10:33:56,19.476389


In [21]:
print("Longer than 1 day: ", (df["duration_minutes"] > 24 * 60).sum())
print("Longer than 7 days:", (df["duration_minutes"] > 7 * 24 * 60).sum())

df.loc[df["duration_minutes"] > 24 * 60, "cause_group"].value_counts()

Longer than 1 day:  379
Longer than 7 days: 42


cause_group
staff               115
rolling stock        72
infrastructure       70
logistical           55
engineering work     35
weather              16
external              9
accidents             7
Name: count, dtype: int64

In [22]:
under_one_day = df["duration_minutes"] <= 24 * 60
print(f"Mean, all rows:            {df['duration_minutes'].mean():.0f} min")
print(f"Mean, rows up to 1 day:    {df.loc[under_one_day, 'duration_minutes'].mean():.0f} min")
print(f"Median, all rows:          {df['duration_minutes'].median():.0f} min")

Mean, all rows:            163 min
Mean, rows up to 1 day:    113 min
Median, all rows:          47 min


- The median disruption lasts **47 minutes**; 75% end within about 2 hours, 90% within about 4.5 hours.
- The mean is more than 3× the median. Even without the disruptions longer than one day, the mean is still
  about 2.4× the median, so the skew is the shape of the **whole** distribution, not just a few outliers.
  Durations should be summarised with the median and percentiles, and plotted on a log scale.
- The longest records are long-running **service changes** rather than single incidents: the Arriva strike in the
  north (Feb 2023, ~28 days), a damaged railway bridge (Roermond–Sittard, ~31 days), multi-week engineering work,
  rolling-stock shortages on cross-border lines. They are believable, so they are **kept**.
- 379 disruptions last more than a day (strikes are the largest group) and 42 more than a week.
- This skew is one reason to predict *long vs. not long* rather than the exact number of minutes:
  with a yes/no target a 28-day strike simply counts as "long" and cannot dominate the model.

## 7. Year boundaries

In [23]:
df["year"] = df["start_time"].dt.year

crosses_year = df["end_time"].dt.year > df["year"]
print("Disruptions ending in a later year:", crosses_year.sum())
df.loc[crosses_year, ["rdt_id", "rdt_lines", "cause_en", "start_time", "end_time", "duration_minutes"]]

Disruptions ending in a later year: 4


,rdt_id,rdt_lines,cause_en,start_time,end_time,duration_minutes
15199,40435,"Amsterdam Centraal - Schiphol Airport, Brussel...",large crowds,2021-12-28 10:33:56,2022-01-16 21:59:29,28046.0
15218,40454,Antwerpen-Centraal - Roosendaal,large crowds,2021-12-29 11:48:05,2022-01-16 21:59:10,26531.0
15257,40493,"Berlin Ostbahnhof - Hengelo, Bielefeld Hbf - H...",problems with the rolling stock,2021-12-31 13:01:34,2022-01-01 00:25:15,684.0
31832,57221,Heerlen - Maastricht,problems with the rolling stock,2024-12-27 07:45:33,2025-01-16 00:02:17,28337.0


Only 4 rows. Three run from December 2021 into January 2022, both training years. One runs from December 2024
(validation) into January 2025 (test).

**Rule:** a disruption belongs to the year it **starts**, because that is when the prediction is made.
With `year` taken from `start_time`, these rows cause no problem for the split and are kept.

## 8. Causes

In [24]:
df["cause_group"].value_counts(dropna=False)

cause_group
rolling stock       16478
infrastructure       8775
external             4600
accidents            3826
logistical           1504
engineering work     1227
staff                1074
unknown               733
weather               352
NaN                     2
Name: count, dtype: int64

In [25]:
# What does the "unknown" group actually contain?
df.loc[df["cause_group"] == "unknown", "cause_en"].value_counts().head()

cause_en
technical investigation       664
cause yet unknown              43
multiple disruptions           20
an obstruction on the line      5
an earlier disruption           1
Name: count, dtype: int64

In [26]:
# The two rows without a cause group
df.loc[df["cause_group"].isna(), ["rdt_id", "cause_en", "statistical_cause_en", "start_time"]]

,rdt_id,cause_en,statistical_cause_en,start_time
29938,55327,deployment of security staff,deployment of security staff,2024-09-05 21:41:49
31443,56832,deployment of security staff,deployment of security staff,2024-12-01 19:34:39


- Rolling stock (broken-down or stranded trains) is by far the largest group, followed by infrastructure.
- "unknown" is mostly `technical investigation` and `cause yet unknown`: **the cause was not known when the message
  was published**. That is real information, and exactly the situation the model will be in at prediction time,
  so "unknown" is kept as its own category. It is not redistributed over other causes, which would invent data.
- The 2 rows without a group are both `deployment of security staff`. They get the group `"external"`.

### `cause_en` vs. `statistical_cause_en`

In [27]:
cause_differs = df["cause_en"] != df["statistical_cause_en"]
print("Rows where the two causes differ:", cause_differs.sum())

df.loc[cause_differs, ["cause_en", "statistical_cause_en"]].value_counts().head(10)

Rows where the two causes differ: 190


cause_en               statistical_cause_en       
an earlier disruption  broken down train              32
                       collision                      24
                       stranded train                 10
                       person on the railway track    10
                       damaged overhead wires          9
                       an emergency call               8
                       power failure                   7
                       signal failure                  7
                       problem in a tunnel             6
                       system failure                  6
Name: count, dtype: int64

In [28]:
df.loc[cause_differs, "cause_en"].value_counts()

cause_en
an earlier disruption    171
multiple disruptions      15
cause yet unknown          4
Name: count, dtype: int64

In all 190 rows where they differ, `cause_en` is vague ("an earlier disruption", "multiple disruptions",
"cause yet unknown") and `statistical_cause_en` names the specific underlying cause. The statistical cause is
kept for statistics, and part of it was only known **after** the message was published.

**Decision:** use `cause_en` and `cause_group` as features. `statistical_cause_*` is not used, to avoid leakage.

**Limitation:** even `cause_en` is the *final* cause. If NS changed the cause while the disruption was running,
only the last version is in the data.

### Dutch vs. English cause names

In [29]:
dutch_names_per_cause = df.groupby("cause_en")["cause_nl"].unique()
dutch_names_per_cause[dutch_names_per_cause.str.len() > 1]

cause_en
damaged overhead wires     [defecte bovenleiding, beschadigde bovenleiding]
damaged railway bridge    [defecte spoorbrug, beschadigd spoorviaduct, b...
strike                                          [staking, acties personeel]
Name: cause_nl, dtype: object

Only 3 English causes have more than one Dutch wording (for example *staking* and *acties personeel* both become
*strike*). The English column is the slightly cleaner one, which is another reason to use `cause_en`.

## 9. Hidden blanks

In [30]:
blank_counts = {}
for col in df.columns:
    if pd.api.types.is_string_dtype(df[col]):
        blank_counts[col] = (df[col].str.strip() == "").sum()

pd.Series(blank_counts)

ns_lines                0
rdt_lines               0
rdt_lines_id            0
rdt_station_names       0
rdt_station_codes       0
cause_nl                0
cause_en                0
statistical_cause_nl    0
statistical_cause_en    0
cause_group             0
dtype: int64

No empty strings hide behind non-missing values: `isna()` already showed every gap.

## 10. Messages with the same start time

In [31]:
same_start = df["start_time"].duplicated(keep=False)
print("Rows sharing a start time with another row:", same_start.sum())

df.loc[same_start, ["rdt_id", "rdt_lines", "cause_en", "start_time", "end_time", "duration_minutes"]].sort_values("start_time").head(10)

Rows sharing a start time with another row: 96


,rdt_id,rdt_lines,cause_en,start_time,end_time,duration_minutes
116,25352,"Almelo - Deventer, Almelo - Hengelo, Amersfoor...",broken down train,2019-01-08 17:56:08,2019-01-08 18:06:17,10.0
117,25353,"Almelo - Deventer, Almelo - Hengelo, Amersfoor...",broken down train,2019-01-08 17:56:08,2019-01-08 18:07:57,12.0
2870,28106,"Amsterdam Centraal - Schiphol Airport, Rotterd...",stranded train,2019-06-24 17:04:49,2019-06-24 18:44:45,100.0
2871,28107,Rotterdam Centraal - Schiphol Airport (HSL),stranded train,2019-06-24 17:04:49,2019-06-24 18:10:17,65.0
3186,28422,Breda - Tilburg,collision with a vehicle,2019-07-13 18:19:02,2019-07-13 19:22:22,63.0
3187,28423,Breda - Tilburg,collision with a vehicle,2019-07-13 18:19:02,2019-07-13 18:56:59,38.0
4250,29486,Rotterdam Centraal - Schiphol Airport (HSL),defective railway track,2019-09-17 11:34:53,2019-09-18 03:41:23,967.0
4251,29487,Amersfoort - Schiphol Airport,defective railway track,2019-09-17 11:34:53,2019-09-18 03:41:13,966.0
4391,29627,"Amersfoort - Schiphol Airport, Lelystad Centru...",damaged overhead wires,2019-09-27 18:00:55,2019-09-28 01:41:09,460.0
4392,29628,Den Haag Centraal - Leiden Centraal,damaged overhead wires,2019-09-27 18:00:55,2019-09-28 01:40:38,460.0


In [32]:
groups = df.loc[same_start].groupby("start_time")
print("Groups:                ", groups.ngroups)
print("  with the same cause: ", (groups["cause_en"].nunique() == 1).sum())
print("  with the same lines: ", (groups["rdt_lines"].nunique() == 1).sum())

Groups:                 48
  with the same cause:  43
  with the same lines:  20


Almost always the same cause, but usually **different lines**: one incident announced as separate messages for
different routes (for example a stranded train near Schiphol: 100 min on one route, 65 min on the other).
Each route recovered at its own time, so these are not duplicates. **Kept.**

## 11. Back-to-back and overlapping messages on the same line

Example from 2025: two "collision" messages on Den Haag HS – Rotterdam Centraal, the second starting 4 seconds
after the first ended. That is probably one incident recorded as two messages. How often does this happen?

For each line, sort by start time and compute the gap between a message's start and the previous message's end.
The calculation is done on a sorted copy, so `df` keeps its original order.

In [33]:
by_line = df.dropna(subset=["rdt_lines"]).sort_values(["rdt_lines", "start_time"])

prev_end = by_line.groupby("rdt_lines")["end_time"].shift(1)
prev_cause = by_line.groupby("rdt_lines")["cause_en"].shift(1)

gap = (by_line["start_time"] - prev_end).dt.total_seconds() / 60
same_cause = by_line["cause_en"] == prev_cause

In [34]:
rows = []
for window in [1, 5, 15]:
    in_window = gap.between(0, window)
    rows.append({
        "gap": f"0 to {window} min",
        "any cause": in_window.sum(),
        "same cause as previous": (in_window & same_cause).sum(),
    })

pd.DataFrame(rows)

,gap,any cause,same cause as previous
0,0 to 1 min,140,103
1,0 to 5 min,311,229
2,0 to 15 min,588,412


In [35]:
overlap = gap < 0
print("Messages starting before the previous one on the same line ended:", overlap.sum())
print("  of which with the same cause:", (overlap & same_cause).sum())
print(f"  median overlap: {-gap[overlap].median():.0f} min")

Messages starting before the previous one on the same line ended: 989
  of which with the same cause: 608
  median overlap: 59 min


- About 230 messages start within 5 minutes of the previous message on the same line with the same cause
  (about 0.6% of rows). These are likely one incident split over two messages.
- A negative gap means two messages were **active at the same time** on the same line, for example a collision
  during long engineering works, or an updated message posted before the old one was closed. That happens in
  real life, so these rows are **kept**.

**Decision:** keep all records as they are. One row = one NS message. Merging chains of messages into incidents
would touch well under 1% of rows and add a lot of complexity; it is noted as a limitation and a v2 idea.

## 12. Apply the cleaning decisions and save

In [36]:
rows_before = len(df)

no_target = df["duration_minutes"].isna()
zero_duration = df["duration_minutes"] == 0

clean = df.loc[~no_target & ~zero_duration].copy()
clean["cause_group"] = clean["cause_group"].fillna("external")
clean = clean.drop(columns=["calculated_duration"]).reset_index(drop=True)

print(f"Rows before:                {rows_before:,}")
print(f"  dropped, no duration:     {no_target.sum():,}")
print(f"  dropped, zero duration:   {zero_duration.sum():,}")
print(f"Rows after:                 {len(clean):,}")

assert len(clean) == rows_before - no_target.sum() - zero_duration.sum()
assert clean["duration_minutes"].gt(0).all()
assert clean["cause_group"].notna().all()

Rows before:                38,571
  dropped, no duration:     111
  dropped, zero duration:   683
Rows after:                 37,777


In [37]:
clean["year"].value_counts().sort_index()

year
2019    5564
2020    4271
2021    4859
2022    5438
2023    5130
2024    5909
2025    6606
Name: count, dtype: int64

In [38]:
processed_dir.mkdir(parents=True, exist_ok=True)
output_path = processed_dir / "disruptions_clean.parquet"
clean.to_parquet(output_path, index=False)

# Read it back to check that the file and the column types survived
check = pd.read_parquet(output_path)
print(output_path.name, check.shape)
check.dtypes

disruptions_clean.parquet (37777, 15)


rdt_id                           int64
ns_lines                           str
rdt_lines                          str
rdt_lines_id                       str
rdt_station_names                  str
rdt_station_codes                  str
cause_nl                           str
cause_en                           str
statistical_cause_nl               str
statistical_cause_en               str
cause_group                        str
start_time              datetime64[us]
end_time                datetime64[us]
duration_minutes               float64
year                             int32
dtype: object

## Findings

**Data:** 38,571 NS disruption messages from 2019–2025. After cleaning, **37,777 rows** are saved to
`data/processed/disruptions_clean.parquet`.

**Cleaning decisions**

| Rule | Action | Rows |
|---|---|---|
| No `duration_minutes` (110 without end time + 1 unclosed 59-day message) | drop: no target | 111 |
| `duration_minutes == 0` (opened and closed within 30 s, mostly 2019–2020) | drop: not a real disruption | 683 |
| `cause_group` missing (`deployment of security staff`) | fill with `"external"` | 2 |
| Missing lines or stations | keep, fill in notebook 03 | 12 / 8 |
| Daylight-saving rows, year-crossing rows, overlaps, same start time | keep | – |

**What I learned about the data**

1. **`duration_minutes` is the correct target.** It is rounded to the nearest minute and handles daylight saving
   time; a naive `end − start` is off by 60 minutes for the 7 disruptions that ran through a clock change.
2. **Durations are heavily right-skewed:** median 47 min, 90th percentile about 4.5 hours, a tail of strikes and
   closures lasting weeks. These long records are real, so a yes/no target ("long or not") is more robust than
   predicting minutes.
3. **Recording practices changed over time:** zero-length messages almost disappear after 2020 and missing end
   times cluster in 2020. Year-to-year comparisons need care.
4. **Leakage risk in causes:** `statistical_cause_*` sometimes contains causes known only later. The model uses
   `cause_en` and `cause_group`; even these are the *final* causes, which is a limitation.
5. **One incident can produce several messages** (different routes at the same second, or back-to-back messages on
   the same line). The unit of analysis stays one NS message; merging is a v2 idea.

**Next:** notebook 02 (EDA): choose the "long disruption" threshold, compare causes, lines and time of day.